# Week 8 — Knowledge Base + Retrieval (RAG Basics)

**Learning Goals:**
- Build a small, student-curated knowledge base
- Implement simple retrieval (BM25 / TF-IDF)
- Ground Copilot outputs in retrieved text

**Deliverables:**
- `kb/` folder with fault_tree.md and glossary.md
- `src/copilot/retriever.py` with BM25 search
- Copilot demo showing KB citations

In [ ]:
import sys
sys.path.insert(0, '../src')

from copilot.retriever import KBRetriever, retrieve_kb
from pathlib import Path

print('Imports OK')

## 1. Explore the Knowledge Base

In [ ]:
# Load and inspect KB files
kb_dir = Path('../kb')
for f in sorted(kb_dir.glob('*.md')):
    content = f.read_text()
    print(f'\n=== {f.name} ({len(content)} chars) ===')
    print(content[:500] + '...')

## 2. Test Retrieval

In [ ]:
# Initialize retriever
retriever = KBRetriever(str(kb_dir))
print(f'KB loaded: {len(retriever.documents)} chunks')

# Test queries
queries = [
    'HPC degradation maintenance recommendation',
    'sensor_13 core speed increasing',
    'when to escalate uncertain prediction',
    'what is NASA scoring function',
    'fan degradation bypass ratio',
]

for q in queries:
    print(f'\n--- Query: "{q}" ---')
    results = retriever.search(q, top_k=2)
    for i, r in enumerate(results):
        print(f'  [{i+1}] Source: {r["source"]}, Score: {r["score"]:.3f}')
        print(f'      {r["text"][:150]}...')

## 3. Copilot with KB Grounding

In [ ]:
from copilot.tools import retrieve_knowledge, recommend_action

# Simulate a prediction + explanation
mock_prediction = {
    'rul_mean': 45.0,
    'rul_std': 8.0,
    'confidence_interval_95': [29.3, 60.7],
    'uncertainty_level': 'medium',
}

mock_explanation = {
    'top_sensors': [
        {'sensor': 'sensor_11', 'importance': 0.25, 'rank': 1},
        {'sensor': 'sensor_15', 'importance': 0.20, 'rank': 2},
        {'sensor': 'sensor_4', 'importance': 0.15, 'rank': 3},
    ],
    'method': 'integrated_gradients',
}

# Retrieve KB
kb_results = retrieve_knowledge('sensor_11 P30 HPC degradation maintenance')
print(f'Retrieved {len(kb_results["snippets"])} KB snippets')

# Generate recommendation
rec = recommend_action(mock_prediction, mock_explanation, kb_results)
print(f'\nAction: {rec["action"]}')
print(f'Recommendation: {rec["recommendation"]}')
print(f'Confidence: {rec["confidence"]}')
print(f'KB Reference: {rec["supporting_evidence"]["kb_reference"][:200]}')

## 4. Groundedness Check

Every recommendation must reference at least 1 retrieved KB snippet.

In [ ]:
# Test that KB citation is always present
test_scenarios = [
    {'rul_mean': 10, 'rul_std': 3, 'uncertainty_level': 'low'},
    {'rul_mean': 50, 'rul_std': 12, 'uncertainty_level': 'medium'},
    {'rul_mean': 100, 'rul_std': 25, 'uncertainty_level': 'high'},
]

for scenario in test_scenarios:
    pred = {**scenario, 'confidence_interval_95': [scenario['rul_mean']-20, scenario['rul_mean']+20]}
    kb = retrieve_knowledge('degradation maintenance')
    rec = recommend_action(pred, mock_explanation, kb)
    
    has_citation = rec['supporting_evidence']['kb_reference'] != 'No KB snippets available.'
    print(f'RUL={scenario["rul_mean"]:3.0f}, Std={scenario["rul_std"]:2.0f}: '
          f'Action={rec["action"]:10s} | KB cited: {"✓" if has_citation else "✗"}')